In [5]:
import os
import torch

`torch.device()` 和 `device_map` 都是 PyTorch 生态中用于设备分配的重要工具, 但它们的应用场景有所不同.

下面我将为您详细解析几种常用的方法, 并提供代码示例, 帮助您根据具体需求选择最合适的方式.

### 方法一: 使用环境变量 `CUDA_VISIBLE_DEVICES` (最推荐的通用方法)

在执行 Python 脚本之前, 通过设置环境变量 `CUDA_VISIBLE_DEVICES` 可以限制程序"可见"的 GPU 设备. 这是一种非常简洁且高效的方式, 可以从根本上隔离 GPU 资源.

**工作原理:**
您可以通过这个环境变量指定物理 GPU 的索引, PyTorch 程序启动后, 只会将这些指定的 GPU 识别为可用设备, 并从 0 开始重新编号.

**如何使用:**
假设您想使用物理索引为 `2` 和 `3` 的两张显卡, 您可以在终端中这样启动您的训练脚本:

```bash
CUDA_VISIBLE_DEVICES=2,3 python your_training_script.py
```

在您的 Python 代码内部, 您就可以像只有两张显卡一样进行操作了:

```python
import torch
import torch.nn as nn

# 检查可用的 GPU 数量, 此时应为 2
print(f"Available GPUs: {torch.cuda.device_count()}")

# 定义您的模型
model = YourModel()

# 使用 nn.DataParallel 来实现数据并行
if torch.cuda.device_count() > 1:
  print(f"Let's use {torch.cuda.device_count()} GPUs!")
  model = nn.DataParallel(model)

# 将模型移动到默认的 GPU 上(此时 cuda:0 对应物理上的 GPU 2)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

# 之后的数据也需要移动到这个设备上
# for data in dataloader:
#     inputs, labels = data
#     inputs, labels = inputs.to(device), labels.to(device)
#     ...
```

**优点:**
* 简洁明了: 无需在 Python 代码中硬编码 GPU 索引.
* 隔离性好: 对代码的侵入性最小, 程序完全感知不到其他 GPU 的存在.
* 灵活性高: 可以轻松地在不修改代码的情况下更换使用的 GPU.

您也可以在 Python 脚本的开头通过 `os` 模块来设置, 但请务必确保在任何 PyTorch CUDA 操作之前执行.

```python
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"

import torch
# ... 后续代码和上面一样
```

### 方法二: 使用 `torch.device` 和 `nn.DataParallel`

如果您不想使用环境变量, 也可以在 PyTorch 代码中直接指定要使用的 GPU 设备. 这通常与 `nn.DataParallel` 模块结合使用.

**工作原理:**
`nn.DataParallel` 可以将模型复制到多个 GPU 上, 并将输入数据在 batch 维度上进行切分, 分别送到不同的 GPU 上进行计算, 最后将结果汇总. 您可以通过 `device_ids` 参数指定使用哪些 GPU.

**如何使用:**
假设您想使用物理索引为 `4` 和 `5` 的两张显卡.

```python
import torch
import torch.nn as nn

# 定义您的模型
model = YourModel()

# 指定要使用的 GPU 索引
device_ids = [4, 5]

# 使用 nn.DataParallel 并指定 device_ids
model = nn.DataParallel(model, device_ids=device_ids)

# 选择一个主设备来汇总结果, 通常是 device_ids 列表中的第一个
# 注意: 即使使用了多个 GPU, 也需要将模型和数据发送到一个主设备上
device = torch.device(f"cuda:{device_ids[0]}" if torch.cuda.is_available() else "cpu")
model.to(device)

# 之后的数据也需要移动到主设备上
# for data in dataloader:
#     inputs, labels = data
#     inputs, labels = inputs.to(device), labels.to(device)
#     ...
```

**要点:**
* `torch.device()` 本身用于指定`一个`设备, 在多 GPU 场景下, 它通常用来指定主设备.
* `nn.DataParallel` 是实现数据并行的关键, 通过 `device_ids` 参数来精确控制使用的 GPU.
* 注意: `nn.DataParallel` 会将数据默认分发到指定的设备上, 但模型本身和最终的输出会集中在主设备(`device_ids[0]`)上, 这可能会导致轻微的负载不均. 对于更高级的分布式训练, 推荐使用 `torch.nn.parallel.DistributedDataParallel` (DDP), 它性能更好, 负载更均衡.

### 方法三: 使用 `device_map` (主要用于 Hugging Face Transformers 和 Accelerate)

`device_map` 参数主要用于模型并行, 特别是当模型非常大, 单张 GPU 无法完全容纳时. 它允许您将模型的不同层放置在不同的设备上(包括不同的 GPU、CPU 甚至硬盘).

**工作原理:**
`device_map` 是一个字典, 它将模型中的模块名称映射到相应的设备索引. Hugging Face的 `Accelerate` 库可以根据硬件情况自动创建这个映射.

**如何使用:**
这通常在加载预训练模型时使用, 例如使用 Hugging Face `transformers` 库.

1. 自动设备映射 (`"auto"`)
   当您设置 `device_map="auto"` 时, `Accelerate` 会自动在所有可见的 GPU 之间分配模型层, 以实现负载均衡. 您可以结合 `CUDA_VISIBLE_DEVICES` 来让它在您指定的两张卡上自动分配.

   ```bash
   CUDA_VISIBLE_DEVICES=6,7 python your_hf_script.py
   ```

   ```python
   from transformers import AutoModelForCausalLM

   # Accelerate会将在可见的GPU 6和7上自动分配模型层
   model = AutoModelForCausalLM.from_pretrained("your_model_name", device_map="auto")
   ```

2. 手动设备映射
   您也可以手动创建一个字典来精确控制每一层的位置. 这在需要精细优化时非常有用.

   ```python
   from transformers import AutoModelForCausalLM

   # 假设我们只想在 GPU 0 和 GPU 1 上分配
   # 注意: 这里的"0"和"1"是PyTorch可见的GPU索引
   device_map = {
       'transformer.word_embeddings': 0,
       'transformer.h.0': 0,
       'transformer.h.1': 0,
       # ... 其他层 ...
       'transformer.h.12': 1,
       'transformer.h.13': 1,
       # ... 其他层 ...
       'lm_head': 1,
   }

   model = AutoModelForCausalLM.from_pretrained("your_model_name", device_map=device_map)
   ```

** `device_map` 总结:**
* 主要用途: 模型并行, 适用于无法在单个 GPU 上加载的大型模型.
* 与数据并行的区别: 数据并行是每个 GPU 上都有一个完整的模型副本, 处理不同的数据批次; 模型并行是模型的不同部分位于不同的 GPU 上, 共同处理同一个数据批次.
* 适用库: 主要与 Hugging Face `transformers` 和 `Accelerate` 库紧密集成.

### 结论与建议

对于您希望在两张显卡上训练模型的需求, 通常指的是**数据并行**, 因此我最推荐以下两种方式:

1. 首选 `CUDA_VISIBLE_DEVICES`: 这是最简单、最灵活且对代码无侵入的方式. 您只需要在启动脚本时设置好环境变量, 代码内部就可以使用标准的 `nn.DataParallel` 或 `nn.DistributedDataParallel` 即可.

2. 备选 `nn.DataParallel` 与 `device_ids`: 如果您需要在代码内部动态控制 GPU 分配, 可以直接使用 `nn.DataParallel(model, device_ids=[i, j])` 的方式.

如果您正在处理一个巨大的模型(例如大型语言模型), 并且单卡显存不足, 那么**模型并行**和 `device_map` 才是您需要考虑的方案.



In [6]:
torch.cuda.is_available()

True

In [7]:
torch.cuda.device_count()

8

In [8]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [9]:
torch.cuda.is_available()

True

In [10]:
torch.cuda.device_count()

2

In [11]:
device = torch.device("cuda")

In [12]:
device

device(type='cuda')